# Highway 97 segment 4: traversal-time EDA and modelling methodology

**Question:** how much of bus traversal time is a recurring calendar pattern, and how much is a persistent, unusual deviation? Start with Route `97-KEL`, direction 0: **Hwy 97 at Westside Rd Stn (103745) -> Queensway Exchange Bay B (102853)**.

This is a separate, offline **EDA and methodology notebook, not a fitted forecasting experiment**. It does not depend on running [dynamics_model.ipynb](dynamics_model.ipynb). Use the `transit_env` kernel and run from the repository root or `notebooks/`. Only a trusted local snapshot is loaded; no database queries, downloads, map tiles or new dependencies are required.

Road traversal is downstream arrival minus upstream departure. It **excludes upstream dwell**, unlike arrival-to-arrival time or signed delay change. Final feed estimates are reference proxies, not independently measured AVL arrivals. A long bus traversal may reflect traffic, signals, holding, detours or feed revisions; it is not automatically a road-congestion incident.

Workflow: audit support -> inspect distributions and time series -> separate weekday/hour patterns -> examine tails and residual persistence -> choose a forecasting task and leakage-safe methodology. All calendar centring and anomaly thresholds below use the full EDA sample and are **descriptive only**.

In [ ]:
from pathlib import Path
import json
import sys

repo_root = next(
    (p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent)
     if (p / 'forecasting' / 'forecasting').is_dir()), None,
)
if repo_root is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
sys.path.insert(0, str(repo_root / 'forecasting'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from forecasting import (
    build_route_segments, get_backtest_dataset, prepare_segment_traversal_eda,
)


## 1. Configuration and exact segment identity

Segment numbering is snapshot- and direction-specific. Resolve and check the stop IDs rather than assuming `4` always describes the same road. Static shape is reference geometry only: historical trips need not match current GTFS trip IDs or that shape.

Use UTC for pairing and fixed-grid lags; Vancouver local time for weekday/hour features. `service_day_start_hour=3` gives a **03:00-cutover service-day proxy** for coverage and later blocking, not a recovered GTFS service date. Local clock weekday remains a separate feature. The November DST transition is preserved in UTC even when the local display clock repeats.

`time_bin_size` controls an EDA aggregation, **not a forecast horizon**. The main observations remain one row per completed journey/stop-pair visit. The support and robust-score thresholds are research controls, not verified incident definitions. HTML/CSV exports overwrite only this notebook's output folder; the cached input is never changed.

In [ ]:
dataset_id = '2026-11-10_2_months'
route_id, direction_id, segment_id = '97-KEL', 0, 4
expected_stop_pair = ('103745', '102853')
display_timezone = 'America/Vancouver'
service_day_start_hour = 3
time_bin_size = '15min'
minimum_calendar_visits = 10
minimum_lag_pairs = 20
positive_outlier_score = 3.0
maximum_episode_gap_minutes = 60
output_dir = repo_root / 'notebooks' / 'visuals' / 'segment_4_traversal_time_eda'
figures = {}


In [ ]:
dataset = get_backtest_dataset(repo_root / 'data', source='cache', dataset_id=dataset_id)
static_dir = dataset.static_data_dir
manifest = json.loads((dataset.directory / 'metadata.json').read_text())
stops = pd.read_csv(static_dir / 'stops.txt', dtype={'stop_id': 'string'})
stop_times = pd.read_csv(static_dir / 'stop_times.txt', dtype={'trip_id': 'string', 'stop_id': 'string'})
trips = pd.read_csv(static_dir / 'trips.txt', dtype={'trip_id': 'string', 'route_id': 'string', 'shape_id': 'string'})
shapes = pd.read_csv(static_dir / 'shapes.txt', dtype={'shape_id': 'string'})
segments = build_route_segments(
    stops, stop_times, trips, shapes, route_id=route_id, trip_direction_id=direction_id,
)
selected_segment = segments.set_index('segment_id').loc[segment_id]
resolved_pair = (str(selected_segment.from_stop_id), str(selected_segment.to_stop_id))
if resolved_pair != expected_stop_pair:
    raise ValueError(f'Snapshot segment identity changed: {resolved_pair}; review the controls.')
display(selected_segment.drop(labels=['geometry']).to_frame('reference segment'))
display(pd.Series({k: manifest[k] for k in (
    'dataset_id', 'trip_rows', 'trip_start_utc', 'trip_end_utc',
)}).to_frame('snapshot provenance'))


## 2. Timing eligibility and coverage

Pair exact directed endpoint IDs within the same reliable dated trip instance and **consecutive numeric realtime sequences**. Missing or nonconsecutively numbered visits are not bridged. Select latest publications first (database ID breaks ties), then check missing times, skipped/cancelled/no-data visits, completion and chronology. Missing latest values are not replaced with older estimates.

Unlike the delay identity cohort, a valid traversal does not require arrival delays, upstream arrival, dwell or a positive feed-inferred scheduled allowance. Invalid dwell and traversal are audited independently. Publication counts are repeated reports, whereas duration counts are individual visits; do not add these different audit stages together.

The loader verifies the pinned snapshot. A two-month recording is not necessarily two months of complete bus coverage. Empty bins can mean no service **or missing observations**; establish scheduled service/exposure independently before assigning a cause. Partial boundary days are flagged, not treated as representative complete days.

In [ ]:
timing = prepare_segment_traversal_eda(
    dataset.trip_updates, selected_segment, route_id=route_id, time_bin_size=time_bin_size,
)
visits = timing.observations.query("traversal_exclusion_reason == 'included'").copy()
visits = visits.sort_values('upstream_departure_time').reset_index(drop=True)
visits['departure_local'] = visits.upstream_departure_time.dt.tz_convert(display_timezone)
local_clock = visits.departure_local.dt.tz_localize(None)
visits['departure_display'] = local_clock
visits['service_day'] = (local_clock - pd.Timedelta(hours=service_day_start_hour)).dt.normalize()
visits['weekday'] = local_clock.dt.dayofweek
visits['weekday_name'] = local_clock.dt.day_name()
visits['hour'] = local_clock.dt.hour
visits['traversal_minutes'] = visits.traversal_seconds / 60
visits['dwell_minutes'] = visits.dwell_seconds / 60
visits['log_traversal'] = np.log(visits.traversal_seconds)
display(timing.audit)
display(visits.traversal_minutes.describe(percentiles=[.05, .5, .9, .95, .99]).to_frame('traversal (min)'))
print(f'{len(visits):,} completed visits on {visits.service_day.nunique()} observed service-day proxies.')
print(f'Unknown dwell among valid traversals: {visits.dwell_seconds.isna().sum():,}')


In [ ]:
recording_days = (
    timing.bins.index.tz_convert(display_timezone).tz_localize(None)
    - pd.Timedelta(hours=service_day_start_hour)
).normalize()
day_index = pd.date_range(recording_days.min(), recording_days.max(), freq='D')
daily = visits.groupby('service_day').agg(
    visits=('traversal_minutes', 'size'), median_minutes=('traversal_minutes', 'median'),
    p90_minutes=('traversal_minutes', lambda s: s.quantile(.9)),
).reindex(day_index)
daily['visits'] = daily.visits.fillna(0).astype(int)
daily['partial_recording_day'] = daily.index.isin([day_index[0], day_index[-1]])
daily.index.name = 'service_day'
display(daily)
coverage_figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.12)
coverage_figure.add_trace(go.Bar(x=daily.index, y=daily.visits, name='Visits'), row=1, col=1)
for column, name in [('median_minutes', 'Median'), ('p90_minutes', 'P90')]:
    coverage_figure.add_trace(go.Scatter(x=daily.index, y=daily[column], name=name, connectgaps=False), row=2, col=1)
coverage_figure.update_yaxes(title_text='Completed visits', row=1, col=1)
coverage_figure.update_yaxes(title_text='Traversal (min)', row=2, col=1)
coverage_figure.update_xaxes(title_text='Local service-day proxy (03:00 cutover)', row=2, col=1)
coverage_figure.update_layout(title='Daily coverage and traversal distribution (boundary days are partial)', height=650)
figures['daily_coverage'] = coverage_figure
display(coverage_figure)


### Is a regular time-series grid supported?

Compare 15-, 30- and 60-minute resolution **before** choosing an aggregate model. Percentages below use the entire 24-hour recording grid, including overnight and partial boundaries, not only scheduled service. A bin median based on one bus is not a well-supported typical traffic estimate.

For prospective interpretation, bin by **upstream departure**, not downstream arrival: arrival binning moves a long traversal later because of the target itself. These retrospective medians are not available at bin start; all included buses must complete and their labels must be received. Leave unsupported bins missing. Do not interpolate overnight, carry a stale bus indefinitely, or compress gaps out of an ACF.

In [ ]:
resolution_rows = []
departures = visits.set_index('upstream_departure_time').traversal_seconds
for frequency in ['15min', '30min', '60min']:
    index = pd.date_range(
        min(timing.bins.index.min(), departures.index.min()).floor(frequency),
        max(timing.bins.index.max(), departures.index.max()).floor(frequency), freq=frequency,
    )
    counts = departures.resample(frequency).count().reindex(index, fill_value=0)
    resolution_rows.append({
        'bin_size': frequency, 'calendar_bins': len(counts),
        'empty_pct': 100 * counts.eq(0).mean(),
        'one_bus_pct': 100 * counts.eq(1).mean(),
        'two_or_more_pct': 100 * counts.ge(2).mean(),
        'median_visits_in_nonempty_bin': counts[counts.gt(0)].median(),
    })
resolution_support = pd.DataFrame(resolution_rows)
display(resolution_support)
grid_index = pd.date_range(
    min(timing.bins.index.min(), departures.index.min()).floor(time_bin_size),
    max(timing.bins.index.max(), departures.index.max()).floor(time_bin_size), freq=time_bin_size,
)
regular = departures.resample(time_bin_size).agg(['median', 'count']).reindex(grid_index)
regular['count'] = regular['count'].fillna(0).astype(int)
regular['median_minutes'] = regular['median'] / 60
assert regular['count'].sum() == len(visits)


## 3. Raw time series, distributions and dwell separation

Look for recurring peaks, day-to-day level changes, isolated extremes and sustained runs. Use the full range first: deleting slow buses as outliers would remove the congestion signal we want to understand. Flag impossible timing separately from valid long traversals.

The timestamp axis displays Vancouver wall clocks; internal UTC identity remains intact at DST. The aggregate line has gaps rather than interpolated missing values. Compare the right tail in minutes with a log-duration view: positive, skewed durations often suit modelling on a log scale or direct quantile modelling. Dwell is a separate outcome, not added to traversal and not proof of a traffic mechanism.

In [ ]:
series_figure = go.Figure(go.Scattergl(
    x=visits.departure_display, y=visits.traversal_minutes, mode='markers', name='Each traversal',
    marker=dict(size=4, opacity=.45), customdata=visits[['trip_instance_key', 'dwell_minutes']],
    hovertemplate='%{x}<br>Traversal: %{y:.2f} min<br>Journey: %{customdata[0]}<br>Upstream dwell: %{customdata[1]:.2f} min<extra></extra>',
))
series_figure.add_trace(go.Scatter(
    x=regular.index.tz_convert(display_timezone).tz_localize(None), y=regular.median_minutes,
    name=f'{time_bin_size} departure-bin median', mode='lines', connectgaps=False,
))
series_figure.update_layout(
    title='Segment 4: completed traversal times', yaxis_title='Traversal (min)', height=550,
    xaxis=dict(title='Upstream departure (Vancouver local)', rangeslider=dict(visible=True)),
    legend=dict(orientation='h', y=1.12), margin=dict(t=100, b=100),
)
figures['traversal_time_series'] = series_figure
display(series_figure)


In [ ]:
distribution_figure = make_subplots(rows=1, cols=2, subplot_titles=('Traversal in minutes', 'Log traversal in seconds'))
distribution_figure.add_trace(go.Histogram(x=visits.traversal_minutes, nbinsx=60, name='Minutes'), row=1, col=1)
distribution_figure.add_trace(go.Histogram(x=visits.log_traversal, nbinsx=60, name='Log seconds'), row=1, col=2)
distribution_figure.update_xaxes(title_text='Traversal (min)', row=1, col=1)
distribution_figure.update_xaxes(title_text='log(traversal seconds)', row=1, col=2)
distribution_figure.update_yaxes(title_text='Visits')
distribution_figure.update_layout(title='Full untrimmed duration distributions', showlegend=False, height=450)
dwell_figure = px.scatter(
    visits, x='dwell_minutes', y='traversal_minutes', color='hour',
    hover_data=['trip_instance_key', 'departure_display'], opacity=.5,
    labels={'dwell_minutes': 'Upstream dwell (min)', 'traversal_minutes': 'Traversal (min)', 'hour': 'Local hour'},
    title='Non-overlapping upstream dwell versus road traversal (association only)',
)
figures['distributions'] = distribution_figure
figures['dwell_vs_traversal'] = dwell_figure
display(distribution_figure, dwell_figure)


## 4. Weekday-by-hour seasonality and changing levels

Inspect the **interaction** between weekday and hour, not just seven daily averages. Rush-hour timing and weekend shape can differ. Show P10/median/P90 and distinct-day support to distinguish changing typical time from changing uncertainty. Individual observations weight the summaries; unequal bus frequency means these are bus-traversal profiles, not time-weighted general-road traffic.

Suppress median cells below `minimum_calendar_visits`; never invent unsupported hours. Eight or so repetitions of each weekday cannot support unrestricted hour-of-week parameters reliably. Later models should pool information with smooth/cyclic calendar effects or a small Fourier basis, and use shrinkage/partial pooling for sparse weekday interactions.

In [ ]:
calendar = visits.groupby(['weekday', 'hour']).agg(
    visits=('traversal_minutes', 'size'), days=('service_day', 'nunique'),
    p10_minutes=('traversal_minutes', lambda s: s.quantile(.1)),
    median_minutes=('traversal_minutes', 'median'),
    p90_minutes=('traversal_minutes', lambda s: s.quantile(.9)),
)
calendar['supported'] = calendar.visits.ge(minimum_calendar_visits)
display(calendar)
hour_counts = calendar.visits.unstack('hour').reindex(index=range(7), columns=range(24)).fillna(0)
hour_medians = calendar.median_minutes.where(calendar.supported).unstack('hour').reindex(index=range(7), columns=range(24))
weekday_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
calendar_figure = make_subplots(rows=2, cols=1, vertical_spacing=.18,
    subplot_titles=('Median traversal (low-support cells blank)', 'Number of completed visits'))
calendar_figure.add_trace(go.Heatmap(z=hour_medians, x=list(range(24)), y=weekday_labels,
    colorscale='YlOrRd', colorbar=dict(title='Minutes', len=.38, y=.8)), row=1, col=1)
calendar_figure.add_trace(go.Heatmap(z=hour_counts, x=list(range(24)), y=weekday_labels,
    colorscale='Blues', colorbar=dict(title='Visits', len=.38, y=.2)), row=2, col=1)
calendar_figure.update_xaxes(title_text='Vancouver local departure hour', dtick=1)
calendar_figure.update_layout(title='Weekday/hour pattern and its evidence', height=750, margin=dict(t=100))
figures['weekday_hour_support'] = calendar_figure
display(calendar_figure)


## 5. Calendar-adjusted deviations and candidate congestion episodes

Centre log traversal within supported weekday/hour cohorts. Scale by **1.4826 times median absolute deviation (MAD)** and flag positive scores above the configured threshold. This is a robust, retrospective diagnostic, **not a calibrated incident probability**. Sparse or zero-MAD cells get an explicitly unavailable score rather than a fallback scale. Quantiles and MADs from small cohorts are uncertain.

A high calendar-adjusted traversal is a candidate for investigation. Check the raw endpoint histories, revisions, vehicle trajectories, exact route/shape variants, holidays, road works and weather before calling it traffic congestion. No anomalous observation is removed here.

Episode grouping below joins **consecutive flagged traversals** only within the same service-day proxy and at most the configured observation gap. The reported span is between observed buses, not verified continuous incident duration. Intervening unflagged/unsupported buses or long gaps break an episode.

In [ ]:
cohorts = visits.groupby(['weekday', 'hour'])
supported = cohorts.traversal_seconds.transform('size').ge(minimum_calendar_visits)
visits['calendar_log_median'] = cohorts.log_traversal.transform('median').where(supported)
visits['log_residual'] = visits.log_traversal - visits.calendar_log_median
visits['absolute_log_residual'] = visits.log_residual.abs()
visits['calendar_log_scale'] = 1.4826 * visits.groupby(['weekday', 'hour']).absolute_log_residual.transform('median')
visits['robust_score'] = visits.log_residual / visits.calendar_log_scale.where(visits.calendar_log_scale.gt(0))
visits['positive_outlier'] = visits.robust_score.gt(positive_outlier_score)
visits['tail_status'] = np.select(
    [visits.robust_score.isna(), visits.positive_outlier], ['Unsupported score', 'Positive calendar outlier'],
    default='Other supported visit',
)
display(visits.tail_status.value_counts().to_frame('visits'))
residual_figure = px.scatter(
    visits, x='departure_display', y='log_residual', color='tail_status',
    hover_data=['trip_instance_key', 'traversal_minutes', 'robust_score'],
    title='Calendar-adjusted log deviations: candidate anomalies, not incident labels',
    labels={'departure_display': 'Upstream departure (Vancouver local)', 'log_residual': 'Log-duration deviation'},
    color_discrete_map={'Positive calendar outlier': 'firebrick', 'Other supported visit': 'steelblue', 'Unsupported score': 'grey'},
)
figures['calendar_residuals'] = residual_figure
display(residual_figure)


In [ ]:
gap_minutes = visits.upstream_departure_time.diff().dt.total_seconds() / 60
same_service_day = visits.service_day.eq(visits.service_day.shift())
close_previous = same_service_day & gap_minutes.gt(0) & gap_minutes.le(maximum_episode_gap_minutes)
new_episode = visits.positive_outlier & ~(visits.positive_outlier.shift(fill_value=False) & close_previous)
visits['episode_id'] = new_episode.cumsum().where(visits.positive_outlier)
episodes = visits.loc[visits.positive_outlier].groupby('episode_id').agg(
    first_departure=('departure_local', 'min'), last_departure=('departure_local', 'max'),
    flagged_visits=('trip_instance_key', 'size'), max_traversal_minutes=('traversal_minutes', 'max'),
    max_robust_score=('robust_score', 'max'),
)
episodes['observed_span_minutes'] = (episodes.last_departure - episodes.first_departure).dt.total_seconds() / 60
display(episodes.sort_values(['flagged_visits', 'max_robust_score'], ascending=False).head(15))
visits['week_start'] = visits.service_day - pd.to_timedelta(visits.service_day.dt.dayofweek, unit='D')
weekly = visits.groupby('week_start').agg(
    visits=('trip_instance_key', 'size'), observed_days=('service_day', 'nunique'),
    scored_visits=('log_residual', 'count'), median_log_residual=('log_residual', 'median'),
    p90_log_residual=('log_residual', lambda s: s.quantile(.9)),
)
display(weekly)
drift_figure = px.line(weekly.reset_index(), x='week_start',
    y=['median_log_residual', 'p90_log_residual'], markers=True,
    labels={'value': 'Calendar-adjusted log deviation', 'week_start': 'Local week start'},
    title='Residual level/tail drift (unequal support and partial weeks remain visible)')
figures['weekly_residual_drift'] = drift_figure
display(drift_figure)


## 6. Does unusual delay persist beyond the calendar pattern?

Compare raw log-duration and calendar-adjusted **bin-median lag correlations** at actual UTC elapsed-time lags. The grid remains complete; pair only two observed bins, report the pair count, and make low-support/constant correlations explicitly unavailable. These are exploratory pairwise correlations, not a classical complete-series ACF or significance test. Adjacent bins often each contain one bus.

Positive short-lag residual correlation supports a dynamic congestion state or AR errors on top of the calendar component. Strong raw daily/weekly correlations alone do not justify an AR model: they can just reflect seasonality. Daily/weekly UTC lags shift local wall-clock alignment across DST; use local calendar regressors for the actual model.

Do not call `dropna()` on the time series before an ACF, or use previous-row lag as though buses were equally spaced. Formal residual ACF/PACF, Ljung-Box tests and decomposition belong **after a calendar model is fitted on training data**, with valid missing-observation handling. ADF/KPSS on raw seasonal, sparse data would not settle model suitability.

In [ ]:
regular['raw_log_median'] = visits.set_index('upstream_departure_time').log_traversal.resample(time_bin_size).median().reindex(grid_index)
regular['residual_log_median'] = visits.set_index('upstream_departure_time').log_residual.resample(time_bin_size).median().reindex(grid_index)
lag_rows = []
for lag_minutes in [15, 30, 60, 120, 1440, 10080]:
    lag_bins = pd.Timedelta(minutes=lag_minutes) / pd.Timedelta(time_bin_size)
    if not float(lag_bins).is_integer():
        continue
    for column in ['raw_log_median', 'residual_log_median']:
        current, previous = regular[column], regular[column].shift(int(lag_bins))
        observed = current.notna() & previous.notna()
        count = int(observed.sum())
        reason = 'available'
        if count < minimum_lag_pairs:
            reason = 'insufficient_pairs'
        elif current[observed].nunique() < 2 or previous[observed].nunique() < 2:
            reason = 'constant_values'
        lag_rows.append({'series': column, 'lag_minutes': lag_minutes, 'pairs': count,
            'correlation': current[observed].corr(previous[observed]) if reason == 'available' else np.nan,
            'status': reason})
lag_correlations = pd.DataFrame(lag_rows)
display(lag_correlations)
lag_figure = px.bar(lag_correlations, x='lag_minutes', y='correlation', color='series', barmode='group',
    hover_data=['pairs', 'status'], title='Observed-bin lag correlations: raw versus calendar-adjusted',
    labels={'lag_minutes': 'UTC elapsed-time lag (minutes)'})
lag_figure.update_xaxes(type='category')
figures['lag_correlations'] = lag_figure
display(lag_figure)


### EDA decision checklist

| Evidence to inspect | Modelling implication |
| --- | --- |
| Missing days, unmatched/unfinished visits, bin support | Fix observation/exposure issues first; retain masks/counts and do not equate no bus with uncongested traffic. |
| Weekday/hour median changes, consistent on multiple days | Use cyclic intraday effects plus weekday interactions or daily/weekly Fourier terms. |
| P90-P10 widening at peaks and right-skewed durations | Model positive durations and conditional uncertainty, not only an additive mean with constant Gaussian variance. |
| Short-lag residual correlation and multi-bus candidate episodes | Add a short-memory residual state with stale-observation decay; calendar-only predictions cannot adapt to current congestion. |
| Isolated extremes without persistent residual correlation | Prefer robust errors/quantiles; audit measurement issues before adding a complex regime model. |
| Weekly residual drift | Allow gradual level changes or rolling refits; compare calendar profiles over time and check schedule/route changes. |
| Apparent anomaly matches a holiday/detour/feed revision | Add independently known covariates or fix labels; statistical tail flags are not validated incident truth. |

Before fitting, also inspect endpoint publication histories and revision size near completion, distinct buses/routes per bin, historical service calendars/shape variants, holiday exposure and weather/road-event timestamps. The current full-pair data does not independently verify historical shape or physical bus identity. Missingness may be informative during congestion; compare coverage in high and low residual periods without assuming causality.

## 7. Which time-series models are suitable?

**First recommendation: a smooth calendar model plus a short-memory dynamic residual**, on log traversal or with a positive predictive distribution. Buses are irregular probes of a traffic state; a pure equally spaced ARIMA is not automatically the best representation. Evaluate the calendar hypothesis before increasing complexity.

$$
\log T_i = \beta_0 + f_{\mathrm{calendar}}(h_i,w_i) + \ell(t_i) + z(t_i) + \epsilon_i.
$$

$T_i$ is traversal in seconds, $t_i$ is upstream-departure time, $h_i$ is local time of day and $w_i$ is local weekday. The smooth calendar term captures repeated patterns, $\ell$ optional slow level drift, $z$ a short-lived congestion deviation, and $\epsilon$ bus-specific variation/measurement error. These components are hypotheses, not quantities identified by this EDA. Calendar forecasts can be made from known planned times; updating $z$ requires **already completed and received** previous traversals, not the current bus's outcome.

| Candidate | Why it fits this problem | Main caveat / when to try |
| --- | --- | --- |
| Weekday/time-slot median or quantile baseline | Transparent recurring pattern and an essential benchmark | Sparse cells need explicit pooling/shrinkage; no adaptation to unexpected congestion. |
| Cyclic GAM or regularized calendar regression | Smooth intraday profile, weekday interactions and interpretable effects | Not dynamic by itself; a good calendar baseline and component of a hybrid. |
| **Dynamic harmonic regression with AR errors / state-space regression** | Low-order daily/weekly Fourier terms plus a short-memory residual and optional level trend | Best initial classical candidate on a justified regular grid; propagate state across missing observations rather than pretending interpolation is evidence. Select Fourier orders and AR order on past-only validation. |
| **Irregular-time state-space residual model** | Calendar prediction plus a latent congestion state that decays according to elapsed time between buses | Natural for per-bus forecasting; explicitly use elapsed time and observation freshness rather than constant previous-row lags. More implementation work than a calendar baseline. |
| SARIMAX | Strong classical comparator for one dominant grid seasonality and short lags; calendar/exogenous inputs can be added | A single seasonal period does not capture both day and week efficiently. A huge hour-of-week seasonal order on sparse 15-minute data is a poor starting point; prefer Fourier regressors to encode the other seasonality. |
| MSTL + ETS / TBATS | Useful regular-grid benchmarks for multiple seasonalities | Need supported regular observations and careful gap handling; decomposition must be train-only. Smoothing can wash out short incidents. |
| Prophet | Convenient daily/weekly/holiday trend baseline | Not inherently a rapid traffic-state updater or an incident detector; changepoints/intervals are not incident probabilities. |
| Lagged-feature boosting with quantile loss | Flexible nonlinear calendar, lag, headway, freshness and covariate interactions | Not a classical time-series model but a useful later comparator. Must construct every lag as-of the forecast origin and validate chronologically. |
| Robust or switching state-space model | Heavy-tailed errors or distinct normal/congested states could describe bursts | Try Student-t/robust innovations first; regime labels can confuse feed errors with traffic. Only add switching after enough independent episodes and out-of-time benefit. |
| LSTM, Transformer, large multi-seasonal neural models | May help with many segments and much more history | Not a first choice for one segment with roughly two months of observations and few repeated weeks. |

Known-ahead calendar effects and latent current congestion are different sources of predictability. A model can respond **after** an incident begins; it cannot reliably anticipate an unobserved accident using weekday/hour alone. Exponentiating a log-scale point forecast usually yields a median, not an arithmetic mean; report quantiles or correctly integrate the predictive distribution when mean time is required.

## 8. Modelling methodology for the next notebook stage

### A. Specify the forecasting contract before fitting

The forecasting task is deliberately **not selected yet**. Good alternatives are:

- **Per bus, at upstream departure:** predict that bus's downstream-arrival minus upstream-departure duration, with median and P10/P90. If the actual departure is not reliably observed in real time, use an operationally available decision timestamp and known/as-of planned departure; retrospective final departures are not automatically available features.
- **Aggregate next-window prediction:** at a fixed decision time, predict median traversal for buses departing in the next 15/30/60-minute window. Define empty-window behavior as no supported target, not zero travel time. This is a different target from individual-bus time; score it separately and report bus support.

For short-horizon experiments, compare explicit horizons such as 15/30/60 minutes, not just one-step-ahead bus index. A 15-minute bin width is not a 15-minute operational forecast horizon.

### B. Define the label and information-availability boundary

1. Reuse exact endpoint/dated-journey eligibility, audit revisions and select a documented truth proxy. Determine when a traversal estimate is sufficiently final; cache-last selection alone does not establish a live finalization rule.
2. Reconstruct features from publications **and ingestion receipt times at or before the forecast origin**. A final label's availability must be no earlier than downstream completion and the receipt/publication times of the endpoint reports used to calculate it. Account for any additional stabilization horizon.
3. Lagged traversal values may use only previously completed, finalized and received visits. Include their age and observation count; expire or decay stale congestion signals. Do not use the current bus's downstream delay/arrival, full-window median, future revisions, or same-visit inferred scheduled allowance as predictors.
4. Calendar inputs use independently known planned timestamps or an as-of observation, not a schedule inferred from the target's downstream outcome. Weather forecasts, incident notices and route changes need their own as-of timestamps. Realized future weather/incident labels are not available predictors.

### C. Split by time and keep tuning away from final evaluation

Use expanding- or rolling-window validation on **whole service-day blocks**, never shuffled rows. As an initial design: at least four observed training weeks, two subsequent seven-day development blocks and a final seven-day evaluation block. Preserve actual calendar gaps; do not squeeze missing days out. Choose an embargo sufficient for the longest label/aggregation/finalization horizon (one day is a conservative starting point for short horizons), and purge overlapping target windows/journeys. Enforce label availability at each training cutoff even with an embargo.

Fit calendar medians, scalers, Fourier/spline parameters, transformations, decomposition and anomaly thresholds **only on each fold's training portion**. Use development blocks to select complexity, freshness limits, pooling and refit cadence. Freeze choices, then evaluate once on a later period covering weekdays and weekends. Decide separately whether online state updates using newly received prior labels are permitted; that is not unrestricted refitting on held-out outcomes.

**This EDA has examined the full cached period.** A last-week split of the same cache would be an exploratory temporal evaluation, not a pristine blind test. For a defensible final result, reserve a **new, untouched future period after design decisions are frozen**. There is no holdout scoring or claimed model skill in this notebook.

### D. Stage the candidates

1. Global training median; pooled weekday/hour median or quantile; last available completed traversal with an explicit freshness limit; previous-day/week same-slot baseline with audited missing support.
2. Smooth/cyclic calendar regression or low-order Fourier regression. Check held-out calendar residuals before adding dynamic state.
3. Calendar plus AR/state-space residual. Compare per-bus elapsed-time state updating against a supported regular-grid representation; keep candidate search small.
4. If justified, add known-as-of headway, upstream conditions, weather/events, robust innovations or quantile boosting. Only then consider a switching congestion regime.

### E. Evaluate usefulness, uncertainty and congestion failures

- **Point accuracy:** MAE in seconds/minutes as primary, RMSE and signed bias as secondary. Avoid relying on MAPE or in-sample R-squared.
- **Probabilistic accuracy:** pinball loss for P10/P50/P90, empirical coverage and width of an 80% interval, and conditional calibration by weekday/hour and congestion state. Nominal interval coverage alone is insufficient if intervals are excessively wide.
- **Stratification:** peak/off-peak, weekday/weekend, supported/sparse cohorts, horizon, lag age, normal periods and training-defined high-residual/tail periods. Report counts and absolute errors, not only an overall average.
- **Congestion behavior:** onset/recovery errors, systematic underprediction during candidate episodes and overprediction after recovery. Confirm episodes externally before reporting incident detection recall or causal traffic effects.
- **Comparison and reliability:** identical eligible forecast origins/targets for every candidate, baseline-relative MAE improvement, daily/weekly error summaries and block-bootstrap uncertainty rather than treating correlated buses as independent samples. Report availability/abstention rates, runtime and freshness.

Adopt the simplest model that consistently beats both calendar and fresh-persistence baselines on out-of-time data, improves peak/tail behavior, and produces useful calibrated uncertainty. Save configuration, dataset checksums, cutoffs, model state and origin-level predictions in a non-overwriting experiment bundle when fitting is introduced.

## 9. Save the EDA evidence

Export standalone HTML with Plotly.js embedded and CSV tables, including the full eligible visits and unsupported residual scores. Files are overwritten on rerun **only in this notebook's output directory**. `snapshot_metadata.json` retains the input manifest/checksums; controls are saved separately. These exports are research evidence, not fitted models or forecast predictions.

Next step: select the operational forecast task and horizon, implement past-only feature/label availability, and fit the staged baselines. Do not reuse the full-sample calendar centring or outlier thresholds above as forecasting inputs.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
for name, figure in figures.items():
    figure.write_html(output_dir / f'{name}.html', full_html=True, include_plotlyjs=True)
tables = {
    'timing_audit': timing.audit, 'eligible_traversals': visits, 'daily_support': daily.reset_index(),
    'resolution_support': resolution_support, 'calendar_profiles': calendar.reset_index(),
    'candidate_episodes': episodes.reset_index(), 'weekly_residuals': weekly.reset_index(),
    'lag_correlations': lag_correlations, 'departure_bins': regular.rename_axis('departure_bin_utc').reset_index(),
}
for name, table in tables.items():
    table.to_csv(output_dir / f'{name}.csv', index=False)
(output_dir / 'snapshot_metadata.json').write_text(json.dumps(manifest, indent=2) + '\n')
pd.Series({
    'dataset_id': dataset_id, 'route_id': route_id, 'direction_id': direction_id, 'segment_id': segment_id,
    'from_stop_id': resolved_pair[0], 'to_stop_id': resolved_pair[1], 'timezone': display_timezone,
    'service_day_start_hour': service_day_start_hour, 'time_bin_size': time_bin_size,
    'minimum_calendar_visits': minimum_calendar_visits, 'minimum_lag_pairs': minimum_lag_pairs,
    'positive_outlier_score': positive_outlier_score, 'maximum_episode_gap_minutes': maximum_episode_gap_minutes,
    'method': 'full_sample_eda_no_model_fitting', 'historical_shape_confirmed': False,
}).to_csv(output_dir / 'eda_controls.csv')
print(f'Saved {len(figures)} HTML figures and EDA tables to {output_dir}')
